In [29]:
import os
import sys 

sys.path.append('./../../audio_chords')
from preprocess_data import load_np, save_np
from train import get_accuracy, \
    PARAMS_MODE, PARAMS_ROOT, PARAMS_MODE


from catboost import CatBoostClassifier
import numpy as np
from functools import partial



# Подготовка данных в 

`harmony/src/audio_chords/preprocess_data.py`

In [6]:
DATA_DIR = '/storage/naumtsevalex/harmony/src/data/root_McGill-Billboard/glinka_chord'
NPY_DIR  = os.path.join(DATA_DIR, 'npy')
CASTOM_NPY_DIR = os.path.join(DATA_DIR, 'castom_npy')
MODEL_DIR = os.path.join(DATA_DIR, 'models')


load_np = partial(load_np, data_dir=CASTOM_NPY_DIR)
save_np = partial(save_np, data_dir=CASTOM_NPY_DIR)

In [3]:
composition_idx = np.load(os.path.join(NPY_DIR, '01_all_chord_labels_extended.npy'))[:, 0].astype(int)
composition_idx[:10]

array([3, 3, 3, 3, 3, 3, 3, 3, 3, 3])

In [4]:
labels = np.load(os.path.join(NPY_DIR, '01_all_chord_labels_extended.npy'))[:, 1:]
labels[:3]

array([[0.        , 0.04643991, 0.        ],
       [0.09287982, 0.13931973, 0.        ],
       [0.13931973, 0.18575964, 0.        ]])

In [5]:
composition_idx_unique = np.unique(composition_idx).astype(int)
composition_idx_unique[:10]

array([ 3,  4,  6, 10, 12, 15, 16, 18, 19, 21])

# Train catboost for harmony prediction (find hyperparams)

In [23]:
# PARAMS_ROOT = {
#     "loss_function": "CrossEntropy",
# }
# PARAMS_MODE = {
#     "loss_function": "CrossEntropy",
# }
# PARAMS_NOCHORD = {
#     "loss_function": "CrossEntropy",
# }


model_roots = CatBoostClassifier(**PARAMS_ROOT, logging_level="Verbose")
model_mode = CatBoostClassifier(**PARAMS_MODE, logging_level="Verbose")
model_nochord = CatBoostClassifier(**PARAMS_NOCHORD, logging_level="Verbose")

In [26]:
load_np = partial(load_np, data_dir=CASTOM_NPY_DIR)

X_train_merged = load_np("X_train_merged")
y_train_merged = load_np("y_train_merged")

X_test_merged = load_np("X_test_merged")
y_test_merged = load_np("y_test_merged")

X_train_shifts_merged = load_np("X_train_shifts_merged")
y_train_chord_merged = load_np("y_train_chord_merged")

X_test_shifts_merged = load_np("X_test_shifts_merged")
# y_test_chord_merged = np.load("y_test_chord_merged")

X_train_modes_merged = load_np("X_train_modes_merged")
y_train_modes_merged = load_np("y_train_modes_merged")

X_test_modes_merged = load_np("X_test_modes_merged")


y_train_nochord_merged = np.where(y_train_merged == 0, 1, 0)
y_test_nochord_merged = np.where(y_test_merged == 0, 1, 0)

In [27]:
model_roots = CatBoostClassifier(**PARAMS_ROOT, logging_level="Verbose")
model_mode = CatBoostClassifier(**PARAMS_MODE, logging_level="Verbose")
model_nochord = CatBoostClassifier(**PARAMS_NOCHORD, logging_level="Verbose")

In [28]:
model_roots.fit(X_train_shifts_merged, y_train_chord_merged)
model_mode.fit(X_train_modes_merged, y_train_modes_merged)
model_nochord.fit(X_train_merged, y_train_nochord_merged)  # TODO: w/o any shifts


0:	learn: 0.6308849	total: 330ms	remaining: 5m 30s
1:	learn: 0.5759106	total: 509ms	remaining: 4m 13s
2:	learn: 0.5253404	total: 682ms	remaining: 3m 46s
3:	learn: 0.4791482	total: 836ms	remaining: 3m 28s
4:	learn: 0.4394696	total: 1.05s	remaining: 3m 29s
5:	learn: 0.4045684	total: 1.25s	remaining: 3m 26s
6:	learn: 0.3722735	total: 1.39s	remaining: 3m 17s
7:	learn: 0.3448268	total: 1.54s	remaining: 3m 10s
8:	learn: 0.3194051	total: 1.7s	remaining: 3m 7s
9:	learn: 0.2962622	total: 1.85s	remaining: 3m 3s
10:	learn: 0.2754436	total: 2.01s	remaining: 3m 1s
11:	learn: 0.2579274	total: 2.19s	remaining: 3m
12:	learn: 0.2426570	total: 2.35s	remaining: 2m 58s
13:	learn: 0.2288732	total: 2.51s	remaining: 2m 56s
14:	learn: 0.2169770	total: 2.68s	remaining: 2m 55s
15:	learn: 0.2054667	total: 2.85s	remaining: 2m 55s
16:	learn: 0.1960186	total: 3.04s	remaining: 2m 55s
17:	learn: 0.1871963	total: 3.23s	remaining: 2m 56s
18:	learn: 0.1786713	total: 3.43s	remaining: 2m 56s
19:	learn: 0.1712061	total: 3.

In [29]:
model_roots.get_feature_importance()

array([ 3.32671044,  1.91423339,  1.04415731,  8.89308737,  1.1661498 ,
        1.28131843,  3.13009426,  1.42344042,  3.10753356,  0.46561235,
        1.89675313,  2.93407818,  4.87461641,  2.35473553,  3.30521323,
       12.28729681,  3.35750011,  2.07304868,  3.3384073 ,  6.95808244,
        5.11418249,  5.36997854, 13.62344749,  6.76032232])

In [32]:
acc_train = get_accuracy(
        model_roots,
        model_mode,
        model_nochord,
        X_train_merged,
        X_train_shifts_merged,
        X_train_modes_merged,
        y_train_merged,
    )
acc_test = get_accuracy(
    model_roots,
    model_mode,
    model_nochord,
    X_test_merged,
    X_test_shifts_merged,
    X_test_modes_merged,
    y_test_merged,
)
print(f"TRAIN ACCURACY: {acc_train}")
print(f"TEST ACCURACY: {acc_test}")

100%|██████████| 54624/54624 [00:44<00:00, 1238.66it/s]


CORRECTLY PREDICTED Ns: 1016 (2.00%


100%|██████████| 13657/13657 [00:05<00:00, 2431.49it/s]

CORRECTLY PREDICTED Ns: 140 (1.10%
TRAIN ACCURACY: 0.8627578078962875
TEST ACCURACY: 0.8398428290766208


In [26]:
X_train_shifts_merged = load_np('X_train_shifts_merged')
X_train_shifts_merged_2 = load_np('X_train_shifts_merged_2')
y_train_chord_merged = load_np('y_train_chord_merged')
X_train_shifts_merged.shape, X_train_shifts_merged_2.shape, y_train_chord_merged.shape

((655488, 24), (655488, 25), (655488,))

In [18]:
X_train_shifts_merged[0]

array([9.32745607e-02, 3.27944037e-02, 1.06440999e-01, 1.26008222e-01,
       4.29205430e-01, 1.86400856e-01, 8.14696296e-04, 0.00000000e+00,
       4.75853960e-02, 1.78287137e+00, 2.83929533e-01, 6.32410615e-03,
       2.48725370e+00, 4.06194333e-02, 2.29822045e-01, 9.36797726e-02,
       1.37707480e+00, 1.10042085e+00, 1.44721070e-02, 1.09504762e+00,
       5.99309741e-02, 2.06245148e+00, 3.30079365e-01, 2.25323119e-01])

In [24]:
# X_train_shifts_merged_2[:24, 24]
X_train_shifts_merged_2[0]

array([9.32745607e-02, 3.27944037e-02, 1.06440999e-01, 1.26008222e-01,
       4.29205430e-01, 1.86400856e-01, 8.14696296e-04, 0.00000000e+00,
       4.75853960e-02, 1.78287137e+00, 2.83929533e-01, 6.32410615e-03,
       2.48725370e+00, 4.06194333e-02, 2.29822045e-01, 9.36797726e-02,
       1.37707480e+00, 1.10042085e+00, 1.44721070e-02, 1.09504762e+00,
       5.99309741e-02, 2.06245148e+00, 3.30079365e-01, 2.25323119e-01,
       6.10240378e-04])

In [31]:
model_roots = CatBoostClassifier(**PARAMS_ROOT, logging_level="Verbose")
model_roots_2 = CatBoostClassifier(**PARAMS_ROOT, logging_level="Verbose")

model_roots.load_model(fname=os.path.join(MODEL_DIR, "model_roots.chpt"))
model_roots_2.load_model(fname=os.path.join(MODEL_DIR, "model_roots_2.chpt"))


In [32]:
preds_root = model_roots.predict_proba(X_train_shifts_merged, verbose=True)
preds_root_2 = model_roots_2.predict_proba(X_train_shifts_merged_2, verbose=True)


In [35]:
preds_root[:3], preds_root_2[:3]

(array([[9.99389760e-01, 6.10240378e-04],
        [9.97435521e-01, 2.56447919e-03],
        [8.07174672e-01, 1.92825328e-01]]),
 array([[9.99725563e-01, 2.74437019e-04],
        [9.98723673e-01, 1.27632711e-03],
        [9.31587706e-01, 6.84122943e-02]]))

In [40]:
# model_roots.plot_tree(tree_idx=0)